In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvaHVkc29uL1JhdEhpbmRsaW1i'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/home/hudson/RatHindlimb/.venv/lib/python3.12/importlib/_bootstrap.py": 1761078637.2165947, "/home/hudson/RatHindlimb/.venv/lib/python3.12/importlib/_bootstrap_external.py": 1761078637.2185946, "/home/hudson/RatHindlimb/.venv/lib/python3.12/zipimport.py": 1761078636.3725944, "/home/hudson/RatHindlimb/.venv/lib/python3.12/codecs.py": 1761078636.0445943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/encodings/aliases.py": 1761078636.6295943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/encodings/__init__.py": 1761078636.6265943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/encodings/utf_8.py": 1761078636.9265945, "/home/hudson/RatHindlimb/.venv/lib/python3.12/abc.py": 1761078636.0015943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/io.py": 1761078636.1395943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/stat.py": 1761078636.2875943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/_collections_abc.py": 1761078635.9655943, "/home/hudson/RatHindlimb/.venv/lib/python3.12/genericpat

In [2]:
import opensim as osim
from pathlib import Path
import polars as pl
import numpy as np
import re
from loguru import logger

osim.Logger.setLevelString('error')
logger.remove()

model_dir = Path('./models/osim')
model_file = model_dir / 'rat_hindlimb.osim'

model = osim.Model(model_file.as_posix())
muscles = model.getMuscles()

with open(model_file, 'r') as file:
    file_content = file.read()

terms = [
    'pelvis', 'femur', 'tibia', 'foot',        # Bodies
    'sacroiliac', 'hip', 'knee', 'ankle'       # Joints
]

for term in terms:
    # Pattern explanation: Find 'term', but ONLY if not followed by '_r'
    file_content = re.sub(rf'{term}(?!_r)', f'{term}_r', file_content)

file_content = file_content.replace('.vtp', '.stl')

# Update muscle names
for muscle in muscles:
    muscle_name = muscle.getName()
    if not muscle_name.startswith('R_'):
      file_content = file_content.replace(muscle_name, 'R_' + muscle_name )

with open(model_file, 'w') as file:
    file.write(file_content)

In [3]:
from src.rathindlimb import model_thelen_to_millard
model = osim.Model(model_file.as_posix())
millard_model = model_thelen_to_millard(model)

out = model_dir / 'rat_hindlimb_millard.osim'
millard_model.printToXML(out.as_posix())

True

In [4]:
from osimpy import OsimGraph
coords_to_lock = ["ankle_r_add", "ankle_r_int", "sacroiliac_r_flx"]
joints = millard_model.getJointSet()
millard_model.initSystem()
graph = OsimGraph(osim_model=millard_model)

for coord_name in coords_to_lock:
  coord = graph.get_coordinate(coord_name)
  coord.set_locked(True)

out = model_dir / 'rat_hindlimb_millard_locked.osim'
graph.osim_model.finalizeFromProperties()
graph.osim_model.finalizeConnections()
graph.osim_model.printToXML(out.as_posix())

True

In [5]:
model = osim.Model(out.as_posix())
graph = OsimGraph(osim_model=model)
results = graph.get_all_muscle_lengths_rom(min_points=10000)

In [6]:
#| eval: false
import plotly.graph_objects as go
figure_dir = Path('./figs')

for muscle_name, muscle_data in results.items():
  fig = go.Figure(
    data=go.Parcoords(
      line=dict(color=muscle_data[muscle_name], colorscale='Viridis', showscale=True),
      dimensions=[
        dict(label=col, values=muscle_data[col]) for col in muscle_data.columns
      ]
    )
  )
  fig.update_layout(title=f'{muscle_name} Lengths')
  # Write to png
  fig.write_image(figure_dir / f'{muscle_name}_orig_lengths.png')

  # TODO: Should this also include moment arms?

In [7]:
#| label: tbl-young-attachments
from IPython.display import Markdown as md
from tabulate import tabulate
import pandas as pd
data = pd.read_csv('./data/attachments/young_model_attachments.csv')

md(tabulate(data, headers='keys', tablefmt='pipe'))

|     | Abbreviation   | Type      | Frame   |     X (mm) |     Y (mm) |   Z (mm) | Notes			                   |
|----:|:---------------|:----------|:--------|-----------:|-----------:|---------:|:------------------|
|   0 | AB             | Origin    | Pelvis  |  17.501    |  -3.087    | 12.02    | nan               |
|   1 | AB             | Insertion | Femur   |  -7.993    |  -3.764    | 17       | nan               |
|   2 | AL             | Origin    | Pelvis  |  10.522    |  -5.42     |  8.854   | nan               |
|   3 | AL             | Insertion | Femur   |  -6.94     |  -3.6      | 15.053   | nan               |
|   4 | AM             | Origin    | Pelvis  |  17.954    |  -2.628    | 13.281   | nan               |
|   5 | AM             | Insertion | Tibia   |  -3        |  -8        | -1.151   | nan               |
|   6 | BFa            | Origin    | Pelvis  |  19.501    |  -0.571    | -5.385   | nan               |
|   7 | BFa            | Via       | Pelvis  |  19.878    |  -6.071    | -2.285   | nan               |
|   8 | BFa            | Via       | Pelvis  |  19.844    |  -8.547    |  1.219   | nan               |
|   9 | BFa            | Insertion | Tibia   |   1.844    |  -8.961    |  2.876   | nan               |
|  10 | BFp            | Origin    | Pelvis  |  22.456    |  -5.732    | -0.362   | nan               |
|  11 | BFp            | Via       | Tibia   |   0.864    | -15.061    |  2.014   | nan               |
|  12 | BFp            | Insertion | Tibia   |   0.048477 | -13.338    | -3.851   | nan               |
|  13 | CF             | Origin    | Pelvis  |  14.103    |  -1.194    | -6.141   | nan               |
|  14 | CF             | Via       | Pelvis  |  13.65     |  -5.582    | -2.7     | nan               |
|  15 | CF             | Via       | Pelvis  |  12.48     |  -8.318    |  0.92    | nan               |
|  16 | CF             | Insertion | Femur   |  -8.599    |  -4.649    | 28.716   | nan               |
|  17 | GI             | Origin    | Pelvis  |  18.827    |  -4.85     | -2.4     | nan               |
|  18 | GI             | Insertion | Femur   |  -2.764    |  -2.573    | -0.624   | nan               |
|  19 | GS             | Origin    | Pelvis  | -12.003    | -10.383    | -1.711   | nan               |
|  20 | GS             | Insertion | Femur   |  -4.996    |  -0.965    |  5.543   | nan               |
|  21 | GMa            | Origin    | Pelvis  |  -7.391    |  -9.731    | -2.277   | nan               |
|  22 | GMa            | Insertion | Femur   |  -9.394    |  -2.368    |  7.021   | nan               |
|  23 | GMe            | Origin    | Pelvis  | -18.604    | -14        | -4.228   | nan               |
|  24 | GMe            | Insertion | Femur   |  -7        |   0.931    | -2.872   | nan               |
|  25 | GMi            | Origin    | Pelvis  | -16.508    | -11.5      | -1.771   | nan               |
|  26 | GMi            | Insertion | Femur   |  -6.595    |   0.979    | -1.597   | nan               |
|  27 | GA             | Origin    | Pelvis  |  19.282    |  -2.5      | 13.354   | nan               |
|  28 | GA             | Insertion | Tibia   |  -3.165    |  -9.661    | -2.262   | nan               |
|  29 | GP             | Origin    | Pelvis  |  20.52     |  -4.773    |  8.583   | nan               |
|  30 | GP             | Insertion | Tibia   |  -3.246    | -12.196    | -3.335   | nan               |
|  31 | IP             | Origin    | Pelvis  | -14.922    | -10.319    |  1.129   | nan               |
|  32 | IP             | Insertion | Femur   |  -3.603    |  -1.128    |  3.751   | nan               |
|  33 | EDL            | Origin    | Femur   | -12.6      |  -4.181    | 31.579   | nan               |
|  34 | EDL            | Via       | Tibia   |   3.324    |  -2.75     |  1.114   | nan               |
|  35 | EDL            | Via       | Tibia   |   0.622    | -39.443    | -9.139   | nan               |
|  36 | EDL            | Via       | Foot    |   4.347    |  -0.954    |  0.908   | nan               |
|  37 | EDL            | Insertion | Foot    |  20.581    |  -4.657    |  1.023   | nan               |
|  38 | LG             | Origin    | Femur   | -12.9      |  -3.824    | 29.6     | nan               |
|  39 | LG             | Via       | TIbia   |   3.517    |  -6.121    |  3.926   | nan               |
|  40 | LG             | Via       | Tibia   |   0.963    | -41.618    | -6.197   | nan               |
|  41 | LG             | Via       | Foot    |  -4.147    |  -1.234    |  1.309   | nan               |
|  42 | LG             | Insertion | Foot    |  -3.09     |  -2.356    |  1.447   | nan               |
|  43 | FDL            | Origin    | Tibia   |  -0.7      |  -6.64     |  2.559   | nan               |
|  44 | FDL            | Via       | Tibia   |  -1.135    | -41.666    | -6.5     | nan               |
|  45 | FDL            | Via       | Foot    |  -4.207    |  -1.497    | -0.06603 | nan               |
|  46 | FDL            | Via       | Foot    |  -1.484    |  -2.881    |  0.159   | nan               |
|  47 | FDL            | Via       | Foot    |   2.079    |  -4.229    | -0.263   | nan               |
|  48 | FDL            | Insertion | Foot    |  19.248    |  -6.962    |  1.465   | nan               |
|  49 | FHL            | Origin    | Tibia   |  -1.427    |  -8.55     |  0.486   | nan               |
|  50 | FHL            | Via       | Tibia   |  -1.952    | -41.859    | -6.38    | nan               |
|  51 | FHL            | Via       | Foot    |  -3.882    |  -1.681    | -0.496   | nan               |
|  52 | FHL            | Via       | Foot    |  -1.153    |  -3.211    | -0.656   | nan               |
|  53 | FHL            | Via       | Foot    |   2.963    |  -4.29     | -0.567   | nan               |
|  54 | FHL            | Insertion | Foot    |  19.135    |  -7.09     | -1.929   | nan               |
|  55 | MG             | Origin    | Femur   |  -5.6      |  -2.026    | 32.495   | nan               |
|  56 | MG             | Via       | Tibia   |  -4.896    |  -6.148    |  1.111   | nan               |
|  57 | MG             | Via       | Tibia   |  -2.257    | -42.328    | -6.557   | nan               |
|  58 | MG             | Via       | Foot    |  -3.545    |  -1.398    | -0.546   | nan               |
|  59 | MG             | Insertion | Foot    |  -3.158    |  -2.639    |  0.201   | nan               |
|  60 | Per            | Origin    | Tibia   |   3.491    |  -5.378    |  2.735   | PerL              |
|  61 | Per            | Via       | Tibia   |   1.375    | -41.618    | -6.197   | nan               |
|  62 | Per            | Via       | Foot    |  -3.636    |  -1.164    |  1.761   | nan               |
|  63 | Per            | Via       | Foot    |  -0.692    |  -2.568    |  2.037   | nan               |
|  64 | Per            | Via       | Foot    |   2.433    |  -3.706    |  2.352   | nan               |
|  65 | Per            | Insertion | Foot    |   6.056    |  -4.665    |  0.846   | PerL              |
|  66 | Pla            | Origin    | Femur   | -12        |  -3.766    | 28.916   | nan               |
|  67 | Pla            | Via       | Tibia   |   2.712    |  -7.046    |  3.975   | nan               |
|  68 | Pla            | Via       | Tibia   |   0.508    | -41.618    | -6.197   | nan               |
|  69 | Pla            | Via       | Foot    |  -4.157    |  -1.334    |  0.796   | nan               |
|  70 | Pla            | Insertion | Foot    |  -3.561    |  -2.715    |  0.819   | nan               |
|  71 | Sol            | Origin    | Tibia   |   0.824    |  -7.027    |  4.31    | nan               |
|  72 | Sol            | Via       | Tibia   |  -0.175    | -41.618    | -6.197   | nan               |
|  73 | Sol            | Via       | Foot    |  -4.129    |  -1.414    |  0.39    | nan               |
|  74 | Sol            | Insertion | Foot    |  -3.561    |  -2.754    |  0.4     | nan               |
|  75 | TA             | Origin    | Tibia   |  -2.308    |  -4.47     | -3.782   | nan               |
|  76 | TA             | Via       | Tibia   |  -2.12     | -14.635    | -7.924   | nan               |
|  77 | TA             | Via       | Tibia   |  -1.431    | -39.204    | -9.245   | nan               |
|  78 | TA             | Via       | Foot    |   4.773    |  -1.062    | -1.312   | nan               |
|  79 | TA             | Insertion | Foot    |   7.945    |  -2.944    | -1.965   | nan               |
|  80 | TP             | Origin    | Tibia   |  -1.427    |  -8.55     |  1.871   | nan               |
|  81 | TP             | Via       | Tibia   |  -1.554    | -42.19     | -6.45    | nan               |
|  82 | TP             | Via       | Foot    |  -4.358    |  -1.609    | -0.337   | nan               |
|  83 | TP             | Via       | Foot    |  -1.159    |  -3.222    | -0.103   | nan               |
|  84 | TP             | Insertion | Foot    |   4.296    |  -4.048    | -1.687   | nan               |
|  85 | Pop            | Origin    | Femur   | -13        |  -3.331    | 30.766   | nan               |
|  86 | Pop            | Via       | Femur   | -12.74     |  -5.68     | 29.888   | nan               |
|  87 | Pop            | Via       | Tibia   |  -0.151    |  -6.323    |  3.876   | nan               |
|  88 | Pop            | Insertion | Tibia   |  -2.611    | -15.257    | -3.901   | nan               |
|  89 | SM             | Origin    | Pelvis  |  21.909    |  -6        |  3.732   | nan               |
|  90 | SM             | Insertion | Tibia   |  -2.89     |  -6.16     | -0.318   | nan               |
|  91 | STa            | Origin    | Pelvis  |  20.964    |  -1.477    | -4.803   | nan               |
|  92 | STa            | Via       | Pelvis  |  21.042    |  -4.671    | -2.5     | nan               |
|  93 | STa            | Via       | Pelvis  |  21.056    |  -6.303    | -0.809   | nan               |
|  94 | STa            | Insertion | Tibia   |  -2.449    | -13.114    | -3.591   | ST?               |
|  95 | STp            | Origin    | Pelvis  |  22.152    |  -5.93     |  0.642   | nan               |
|  96 | STp            | Insertion | Tibia   |  -2.449    | -13.114    | -3.591   | nan               |
|  97 | VI             | Origin    | Femur   |  -9.016    |  -1.543    |  8.885   | nan               |
|  98 | VI             | Via       | Femur   | -12.538    |  -0.655    | 33.545   | nan               |
|  99 | VI             | Via       | Tibia   |  -0.29     |   0.756    | -2.082   | nan               |
| 100 | VI             | Insertion | Tibia   |  -0.527    |  -3.598    | -5       | Tibial tuberosity |
| 101 | VL             | Origin    | Femur   |  -8.878    |  -2.021    |  1.648   | nan               |
| 102 | VL             | Via       | Femur   | -12.657    |  -2.802    | 32.316   | nan               |
| 103 | VL             | Via       | Tibia   |   1.159    |   1.022    | -1.799   | nan               |
| 104 | VL             | Insertion | Tibia   |  -0.527    |  -3.598    | -5       | nan               |
| 105 | VM             | Origin    | Femur   |  -4        |  -1.169    |  4.662   | nan               |
| 106 | VM             | Via       | Femur   |  -7.386    |   0.036067 | 33.844   | nan               |
| 107 | VM             | Via       | Tibia   |  -3.916    |   0.713    | -2.615   | nan               |
| 108 | VM             | Insertion | Tibia   |  -0.527    |  -3.598    | -5       | nan               |
| 109 | OE             | Origin    | Pelvis  |  15.777    |  -6.5      |  4.619   | nan               |
| 110 | OE             | Insertion | Femur   |  -2.809    |  -2.8      |  0.605   | nan               |
| 111 | OI             | Origin    | Pelvis  |  19.884    |  -6.8      |  3.25    | nan               |
| 112 | OI             | Via       | Femur   |  -4.066    |  -1.597    | -1.22    | nan               |
| 113 | Pec            | Origin    | Pelvis  |  20.879    |  -3.121    | 12.704   | nan               |
| 114 | Pec            | Insertion | Femur   |  -7.5      |  -4.5      |  9.344   | nan               |
| 115 | Pir            | Origin    | Pelvis  |  -5.732    | -10.422    | -3.162   | nan               |
| 116 | Pir            | Insertion | Femur   |  -6.145    |   1.521    | -3.3     | nan               |
| 117 | QF             | Origin    | Pelvis  |  20.123    |  -6        |  6       | nan               |
| 118 | QF             | Insertion | Femur   |  -6.201    |  -3.976    | -0.436   | nan               |
| 119 | RF             | Origin    | Pelvis  |   2.081    | -12        |  1       | nan               |
| 120 | RF             | Via       | Femur   |  -9.664    |   0.916    | 33.229   | nan               |
| 121 | RF             | Via       | Tibia   |  -1.716    |   0.815    | -2.518   | nan               |
| 122 | TFL            | Origin    | Pelvis  |  -9.594    |  -9.968    | -0.533   | nan               |
| 123 | TFL            | Insertion | Femur   |  -9.139    |  -0.491    | 20.484   | nan               |

In [8]:
from src.rathindlimb.registration import register_meshes, apply_transformation_to_mesh

# The foot mesh used in the Animatlab model is missing the phalanges, but there is a mesh that includes them that lines up with the one included in the model
mesh_dir = Path('./meshes')

source_file = mesh_dir / 'foot3_r_young_no_phalanges.stl'
target_file = mesh_dir / 'foot2_r_young.stl'
output_file1 = mesh_dir / 'foot_r_young_no_phalanges.stl' 

foot_file = mesh_dir / 'foot3_r_young.stl'
final_file = mesh_dir / 'foot_r_young.stl' 
transform = register_meshes(source_file, target_file, output_file1, seed=123)

apply_transformation_to_mesh(foot_file, transform, final_file)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading and preparing meshes...
Preprocessing point clouds and computing features...


Performing global registration (RANSAC)...


[Open3D WARNING] Too few correspondences (130) after mutual filter, fall back to original correspondences.


Refining registration (ICP)...

Registered mesh saved to meshes/foot_r_young_no_phalanges.stl
Transformed mesh saved to meshes/foot_r_young.stl


TriangleMesh with 73944 points and 24648 triangles.

In [9]:
# Define the source, target, and output files
source_postfix = '_young.stl'
target_postfix = '_johnson.stl'
output_postfix = '.stl'
output_dir = model_dir / 'Geometry'

meshes = ['spine', 'pelvis_r', 'femur_r', 'tibia_r', 'foot_r']
transforms = {}
for mesh in meshes:
    source_file = mesh_dir / (mesh + source_postfix)
    target_file = mesh_dir / (mesh + target_postfix)
    output_file = output_dir / (mesh + output_postfix)
    transforms[mesh] = register_meshes(source_file, target_file, output_file, seed=120)

Loading and preparing meshes...


Preprocessing point clouds and computing features...


Performing global registration (RANSAC)...


Refining registration (ICP)...

Registered mesh saved to models/osim/Geometry/spine.stl
Loading and preparing meshes...


Preprocessing point clouds and computing features...


...

[Open3D WARNING] Too few correspondences (43) after mutual filter, fall back to original correspondences.


Refining registration (ICP)...

Registered mesh saved to models/osim/Geometry/tibia_r.stl
Loading and preparing meshes...


Preprocessing point clouds and computing features...


Performing global registration (RANSAC)...


Refining registration (ICP)...

Registered mesh saved to models/osim/Geometry/foot_r.stl


In [10]:
from src.rathindlimb.registration import convert_points_between_meshes
from loguru import logger

attachment_dir = Path('./data/attachments')
young_attachments = pl.read_csv(attachment_dir / 'young_model_attachments.csv')
young_attachments = young_attachments.with_columns([
    pl.col('X (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
    pl.col('Y (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
    pl.col('Z (mm)').cast(pl.String).str.strip_chars().cast(pl.Float64),
])
# Remove all wrap objects from the bodies
body_set: osim.BodySet = model.getBodySet()
n_bodies = body_set.getSize()
for i in range(n_bodies):
    body = model.getBodySet().get(i)
    # Remove all wrap objects from the body
    wrap_set: osim.WrapObjectSet = body.upd_WrapObjectSet()
    wrap_set.clearAndDestroy()

# Add new path points to the muscles
muscles : osim.SetMuscles = model.getMuscles()
for i in range(muscles.getSize()):
    muscle : osim.Muscle = muscles.get(i)
    muscle_name = muscle.getName()
    
    geo_path: osim.GeometryPath = muscle.getGeometryPath()
    path_points: osim.PathPointSet = geo_path.getPathPointSet()
    existing_path_points = path_points.getSize()
    
    path_wraps: osim.PathWrapSet = geo_path.getWrapSet()
    path_wraps.clearAndDestroy()
    
    # Add the new path points
    muscle_rows = young_attachments.filter(pl.col('Abbreviation') == muscle_name.split('R_')[-1])
    index = 1
    for row in muscle_rows.iter_rows(named=True):
        lower_frame = row['Frame'].lower()
        frame = lower_frame + '_r' if lower_frame != 'spine' else lower_frame
        mesh = model.getBodySet().get(frame)
        point_name = muscle_name + '_' + frame + '_' + row['Type'].lower() + '_' + str(index)
        young_loc = np.array([[row['X (mm)'], row['Y (mm)'], row['Z (mm)']]])/100 # Young has a weird scale
        loc = convert_points_between_meshes(young_loc, transforms[frame])
        vec = osim.Vec3(loc[0][0], loc[0][1], loc[0][2])
        muscle.addNewPathPoint(point_name, mesh, vec)
        index += 1
    # Remove the old path points
    for j in range(existing_path_points - 1, -1, -1):
        path_points.remove(j)

model.finalizeFromProperties()
model.finalizeConnections()
out = model_dir / 'rat_hindlimb_millard_y2j.osim'
model.printToXML(out.as_posix())

True

In [11]:
# TODO: Derive this programmatically
new_frame = [-0.00057, 0.0399598, 0.0038162]
rotation2 = [-0.087266499999999996851, -0.075049199999999996469, -0.064577200000000001268, -0.052359900000000000886, -0.040142600000000000504, -0.029670599999999998364, -0.017453300000000001452, -0.0052359900000000002621, 0.0052359900000000002621, 0.017453300000000001452, 0.029670599999999998364, 0.040142600000000000504, 0.052359900000000000886, 0.064577200000000001268]
rotation3 = [0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393, 0.26179900000000000393]
translation1 = [-0.0052385299999999999226, -0.0046464799999999997077, -0.0040425699999999996706, -0.0034725400000000000884, -0.0029796499999999999202, -0.0025907899999999999333, -0.0022994299999999998768, -0.0021100099999999998371, -0.001995639999999999914, -0.0019320699999999999246, -0.0018704800000000001026, -0.0017793500000000000722, -0.0016382700000000000474, -0.0014133999999999999862]
translation2 = [-0.034168400000000001548, -0.03425389999999999685, -0.034185599999999996546, -0.033972299999999996944, -0.033651100000000003232, -0.033278500000000002523, -0.032889799999999996816, -0.032550400000000000167, -0.032282499999999998697, -0.032116100000000001591, -0.032063200000000000034, -0.032110500000000000154, -0.032239700000000003077, -0.032414400000000002933]
translation3 = [0.0026030200000000001601, 0.0028034100000000001032, 0.0028536500000000001101, 0.002903750000000000081, 0.0028935800000000001971, 0.0027731800000000000894, 0.0027023500000000000645, 0.0026110500000000001763, 0.002469499999999999907, 0.0024171399999999999136, 0.0024041000000000001605, 0.0023809000000000000302, 0.002496410000000000122, 0.0026910400000000000119]

model = osim.Model(out.as_posix())
model.initSystem()
knee = osim.CustomJoint.safeDownCast(model.getJointSet().get('knee_r'))
tibia_offset = osim.PhysicalOffsetFrame.safeDownCast(knee.getChildFrame())
tibia_offset.set_translation(osim.Vec3(new_frame[0], new_frame[1], new_frame[2]))

spatial_transform = knee.get_SpatialTransform()
rot2 = spatial_transform.get_rotation2()
rot3 = spatial_transform.get_rotation3()
rot3.set_function(osim.Constant(rotation3[0]))
trans1 = spatial_transform.get_translation1()
simm1 = osim.SimmSpline.safeDownCast(trans1.get_function())
for i, value in enumerate(translation1):
    simm1.setY(i, value)
trans2 = spatial_transform.get_translation2()
simm2 = osim.SimmSpline.safeDownCast(trans2.get_function())
for i, value in enumerate(translation2):
    simm2.setY(i, value)
trans3 = spatial_transform.get_translation3()
simm3 = osim.SimmSpline.safeDownCast(trans3.get_function())
for i, value in enumerate(translation3):
    simm3.setY(i, value)

model.finalizeFromProperties()
model.finalizeConnections()
out = model_dir / 'rat_hindlimb_millard_y2j_knee_fixed.osim'
model.printToXML(out.as_posix())

True

In [12]:
#TODO: Visualize
marker_set_path = model_dir / 'rat_hindlimb_unilateral_markerset.xml'
marker_set = osim.MarkerSet(marker_set_path.as_posix())
model = osim.Model(out.as_posix())
model.getMarkerSet().clearAndDestroy()
model.updateMarkerSet(marker_set)
out = model_dir / 'rat_hindlimb_millard_y2j_knee_fixed_markers.osim'
model.printToXML(out.as_posix())

True

In [13]:
#| label: tbl-johnson-params
parameter_dir = Path('./data/parameters')
johnson_params = pl.read_csv(parameter_dir / 'johnson_2011_parameters.csv')

md(tabulate(johnson_params.to_pandas(), headers='keys', tablefmt='pipe'))

|    | Abbreviation   | Muscle Name               |   Mass (mg) |   ± |   l0 (mm) |   ±_duplicated_0 |   θ0 (deg) |   ±_duplicated_1 |   P0 (g) arch |   ±_duplicated_2 |   Fo (N) |   ±_duplicated_3 |   P0 (g) phy |   ±_duplicated_4 |   vmax (mm/s) arch |   ±_duplicated_5 |   vmax (mm/s) phys |   ±_duplicated_6 |   lts (mm) | ±_duplicated_7   |
|---:|:---------------|:--------------------------|------------:|----:|----------:|-----------------:|-----------:|-----------------:|--------------:|-----------------:|---------:|-----------------:|-------------:|-----------------:|-------------------:|-----------------:|-------------------:|-----------------:|-----------:|:-----------------|
|  0 | AB             | adductor brevis           |        1087 | 157 |      22.8 |              0.4 |         12 |                5 |          1246 |               80 |   12.21  |            0.784 |          nan |              nan |                406 |               21 |                nan |              nan |        0   | 0                |
|  1 | AL             | adductor longus           |          94 |  19 |      16.6 |              2.1 |          9 |                4 |           106 |               15 |    1.039 |            0.147 |          nan |              nan |                 93 |               17 |                nan |              nan |        4.5 | 0.3              |
|  2 | AM             | adductor magnus           |         280 |  93 |      28.4 |              1   |          0 |                0 |           308 |               24 |    3.02  |            0.235 |          nan |              nan |                506 |               32 |                nan |              nan |        0   | 0                |
|  3 | BFa            | biceps femoris anterior   |         470 |  99 |      39.7 |              3.9 |          0 |                1 |           346 |               23 |    3.39  |            0.225 |          nan |              nan |                706 |               60 |                nan |              nan |        0   | 0                |
|  4 | BFp            | biceps femoris posterior  |        1655 | 348 |      37.6 |              1.7 |          7 |                5 |          1274 |               83 |   12.49  |            0.813 |          nan |              nan |                669 |               71 |                nan |              nan |        0   | 0                |
|  5 | CF             | caudofemoralis            |         275 |  91 |      37.9 |              3.2 |          0 |                0 |           210 |               25 |    2.06  |            0.245 |          nan |              nan |                675 |               77 |                nan |              nan |        0   | 0                |
|  6 | EDL            | extensor digitorum longus |         146 |  14 |      13.7 |              1   |         10 |                3 |           225 |                8 |    2.21  |            0.078 |          265 |               65 |                243 |               15 |                205 |               13 |        9   | 0.4              |
|  7 | EHL            | extensor hallucis longus  |          13 |   2 |      12.1 |              0.8 |          5 |                3 |            25 |                2 |    0.25  |            0.02  |          nan |              nan |                216 |               22 |                nan |              nan |       13.6 | 1.2              |
|  8 | FDL            | flexor digitorum longus   |         470 |  53 |       9   |              0.5 |         11 |                5 |          1124 |              217 |   11.02  |            2.127 |         1344 |              494 |                160 |               33 |                140 |               17 |        8.7 | 0.8              |
|  9 | FHL            | flexor hallucis longus    |          22 |   1 |      10.8 |              1   |          2 |                4 |            45 |                1 |    0.44  |            0.01  |          nan |              nan |                193 |                6 |                nan |              nan |        9.7 | 0.0              |
| 10 | GI             | gemellus inferior         |          55 |   2 |       4.2 |              0.2 |         18 |                7 |           291 |               38 |    2.85  |            0.372 |          nan |              nan |                 74 |                9 |                nan |              nan |        0   | 0                |
| 11 | GS             | gemellus superior         |          12 |   2 |       6.1 |              1.1 |         10 |               14 |            38 |               13 |    0.37  |            0.127 |          nan |              nan |                109 |               31 |                nan |              nan |        0   | 0                |
| 12 | GMa            | gluteus maximus           |         793 |  33 |      19.6 |              4.7 |         12 |               10 |           942 |              276 |    9.23  |            2.705 |          nan |              nan |                349 |               75 |                nan |              nan |        4.8 | 0.4              |
| 13 | GMe            | gluteus medius            |        1444 | 182 |      19.9 |              2.7 |         12 |                2 |          1912 |              233 |   18.74  |            2.283 |          nan |              nan |                355 |               35 |                nan |              nan |        0   | 0                |
| 14 | GMi            | gluteus minimus           |         378 |  25 |      11.8 |              0.3 |         16 |                9 |           754 |               61 |    7.39  |            0.598 |          nan |              nan |                211 |               22 |                nan |              nan |        0   | 0                |
| 15 | GA             | gracilis anterior         |         168 |  32 |      30.3 |              4.9 |          0 |                0 |           158 |               10 |    1.55  |            0.098 |          nan |              nan |                539 |               50 |                nan |              nan |        0   | 0                |
| 16 | GP             | gracilis posterior        |         246 |  43 |      29.3 |              4.1 |          3 |                4 |           240 |               35 |    2.35  |            0.343 |          nan |              nan |                163 |               27 |                nan |              nan |        0.9 | 0.1              |
| 17 | IP?            | iliacus                   |         999 |  77 |      22.3 |              0.9 |          7 |                4 |          1043 |               84 |   10.22  |            0.823 |          nan |              nan |                402 |               50 |                nan |              nan |      nan   |                  |
| 18 | IP             | illiopsoas                |        1179 | 126 |      23.8 |              3.3 |         15 |                3 |          1151 |              157 |   11.28  |            1.539 |          nan |              nan |                418 |               64 |                nan |              nan |        4.1 | 0.0              |
| 19 | LG             | lateral gastrocnemius     |         805 | 161 |      12.5 |              0.8 |          9 |                6 |          1863 |              116 |   18.26  |            1.137 |         1734 |               43 |                223 |               17 |                209 |               17 |        9.4 | 0.3              |
| 20 | MG             | medial gastrocnemius      |         720 |  56 |      14.6 |              0.4 |         13 |                4 |          1188 |               92 |   11.64  |            0.902 |         1364 |               30 |                260 |               25 |                238 |                6 |        9.1 | 0.3              |
| 21 | OE             | obturator externus        |         194 |  13 |       6   |              0.8 |         16 |                6 |           732 |               80 |    7.17  |            0.784 |          nan |              nan |                106 |               10 |                nan |              nan |        0   | 0                |
| 22 | OI             | obturator internus        |          54 |  11 |       4.5 |              0.7 |          7 |                6 |           307 |               35 |    3.01  |            0.343 |          nan |              nan |                 81 |               10 |                nan |              nan |        2.8 | 4.0              |
| 23 | Pec            | pectineus                 |         233 |  17 |      10.9 |              0.7 |          7 |                1 |           503 |               14 |    4.93  |            0.137 |          nan |              nan |                193 |                2 |                nan |              nan |        0   | 0                |
| 24 | PerB           | peroneus brevis           |          88 |  11 |       8.6 |              0.9 |          6 |                7 |           243 |               20 |    2.38  |            0.196 |          501 |               40 |                152 |               16 |                117 |                4 |       10.3 | 0.2              |
| 25 | Per            | peroneus longus           |         131 |  16 |       7.6 |              0.3 |          9 |                2 |           404 |               32 |    3.96  |            0.314 |          622 |              122 |                135 |               12 |                113 |                7 |       12.7 | 0.1              |
| 26 | PerQa          | peroneus digiti quarti    |          31 |   4 |       8.8 |              1.9 |          7 |                5 |            86 |               15 |    0.84  |            0.147 |          139 |                2 |                157 |               36 |                116 |               19 |        9.1 | 1.4              |
| 27 | PerQi          | peroneus digiti quinti    |          21 |   3 |       8.6 |              0.8 |          4 |                1 |            56 |                3 |    0.55  |            0.029 |          113 |               22 |                152 |               13 |                116 |                4 |       19.6 | 0.9              |
| 28 | Pir            | piriformis                |         478 |  60 |      10.8 |              1.6 |         16 |                8 |          1137 |              123 |   11.14  |            1.205 |          nan |              nan |                193 |               19 |                nan |              nan |        0   | 0                |
| 29 | Pla            | plantaris                 |         308 |  27 |      12.5 |              1.5 |         11 |                4 |           572 |               56 |    5.61  |            0.549 |          792 |               13 |                223 |               17 |                207 |               13 |        6.7 | 0.2              |
| 30 | Pop            | popliteus                 |          90 |   5 |       6.5 |              0.6 |         13 |                6 |           298 |               17 |    2.92  |            0.167 |          nan |              nan |                115 |                5 |                nan |              nan |        0   | 0                |
| 31 | QF             | quadratus femoris         |         155 |  14 |      11   |              1   |          3 |                2 |           306 |               18 |    3     |            0.176 |          nan |              nan |                197 |               15 |                nan |              nan |        0   | 0                |
| 32 | SM             | semimembranosus           |        1291 | 320 |      35.7 |              1   |          0 |                0 |          1043 |               69 |   10.22  |            0.676 |          nan |              nan |                634 |               57 |                nan |              nan |        0   | 0                |
| 33 | STa            | semitendinosus accessory  |         177 |  25 |      34.1 |              6.4 |          0 |                0 |           142 |               23 |    1.39  |            0.225 |          nan |              nan |                607 |               95 |                nan |              nan |        0   | 0                |
| 34 | STp            | semitendinosus principal  |         707 | 101 |      43.6 |             10.4 |          0 |                0 |           463 |              148 |    4.54  |            1.45  |          nan |              nan |                775 |              193 |                nan |              nan |        0   | 0                |
| 35 | Sol            | soleus                    |         155 |  27 |      16   |              5.1 |          4 |                9 |           145 |               27 |    1.42  |            0.265 |          234 |               29 |                 89 |               22 |                 93 |               11 |        9.5 | 0.5              |
| 36 | TFL            | tensor fascia latae       |         362 |  15 |      20.6 |              0.6 |         11 |                8 |           397 |               30 |    3.89  |            0.294 |          nan |              nan |                367 |               27 |                nan |              nan |        0   | 0                |
| 37 | TA             | tibialis anterior         |         542 |  60 |      15.2 |              0.7 |          9 |                4 |           878 |               81 |    8.6   |            0.794 |         1004 |               45 |                271 |               29 |                290 |                6 |       11.9 | 1.5              |
| 38 | TP             | tibialis posterior        |          85 |   5 |       5.4 |              1.4 |          5 |                3 |           361 |               83 |    3.54  |            0.813 |          605 |               51 |                 97 |               31 |                 68 |                1 |       12.4 | 0.2              |
| 39 | RF             | rectus femoris            |         900 |  67 |      12.2 |              0.9 |         11 |                5 |          1681 |               86 |   16.47  |            0.843 |         1859 |               71 |                216 |               19 |                227 |               13 |        6.6 | 0.8              |
| 40 | VI             | vastus intermedius        |         181 |  12 |      11.6 |              0.6 |         14 |                3 |           260 |               25 |    2.55  |            0.245 |          nan |              nan |                 65 |                9 |                nan |              nan |        6.6 | 0.8              |
| 41 | VL             | vastus lateralis          |        1113 | 170 |      19.8 |              0.3 |         14 |                3 |          1517 |               79 |   14.87  |            0.774 |          nan |              nan |                352 |               25 |                nan |              nan |        6.6 | 0.8              |
| 42 | VM             | vastus medialis           |         396 |  26 |      15.6 |              1.3 |         11 |                6 |           620 |               73 |    6.08  |            0.715 |          611 |               87 |                278 |               34 |                161 |                3 |        6.6 | 0.8              |

In [14]:
#| label: tbl-eng-params
eng_params = pd.read_csv(parameter_dir / 'eng_2008_parameters.csv')

md(tabulate(eng_params, headers='keys', tablefmt='pipe'))

|    | Abbreviation   | Muscle Name               |   Mass (mg) |      ± |   Fiber length (cm) |   ±.1 |   Pennation angle (deg) |   ±.2 |   Muscle length (cm) |   ±.3 |   PCSA (cm2) |   ±.4 |   Lf/Lm |   ±.5 |
|---:|:---------------|:--------------------------|------------:|-------:|--------------------:|------:|------------------------:|------:|---------------------:|------:|-------------:|------:|--------:|------:|
|  0 | AB             | Adductor brevis           |       66.83 |   3.26 |                1.16 |  0.07 |                    0    |  0    |                 1.57 |  0.09 |         0.06 |  0    |    0.75 |  0.02 |
|  1 | AL             | Adductor longus           |      235.5  |  11.86 |                1.07 |  0.08 |                    4.2  |  1.7  |                 1.62 |  0.06 |         0.21 |  0.01 |    0.66 |  0.04 |
|  2 | AM             | Adductor magnus           |      649.83 |  16.49 |                2.21 |  0.08 |                    0    |  0    |                 2.98 |  0.09 |         0.28 |  0.01 |    0.74 |  0.02 |
|  3 | nan            | Adductor tertius          |      685.17 |  18.37 |                1.98 |  0.04 |                    0.6  |  0.6  |                 2.47 |  0.07 |         0.33 |  0.01 |    0.8  |  0.02 |
|  4 | BF             | Biceps femoris            |     2670.83 |  95.48 |                3.4  |  0.06 |                    3.6  |  3.5  |                 5.1  |  0.08 |         0.74 |  0.03 |    0.67 |  0.02 |
|  5 | nan            | Dorsiflexors              |      271.17 |  15.9  |                1.39 |  0.13 |                    9.17 |  2.05 |                 2.49 |  0.38 |         0.16 |  0.11 |    0.58 |  0.06 |
|  6 | EDL            | Extensor digitorum longus |      131.83 |   3.24 |                1.34 |  0.1  |                    9    |  1.1  |                 2.81 |  0.09 |         0.09 |  0.01 |    0.48 |  0.05 |
|  7 | EHL            | Extensor hallucis longus  |       19.5  |   0.67 |                1.18 |  0.07 |                    5.7  |  1.6  |                 1.74 |  0.09 |         0.02 |  0    |    0.68 |  0.02 |
|  8 | FDL            | Flexor digitorum longus   |      463.83 |  10.87 |                0.97 |  0.06 |                   19.3  |  1.9  |                 3.22 |  0.05 |         0.44 |  0.03 |    0.3  |  0.02 |
|  9 | FHL            | Flexor hallucis longis    |       29.67 |   1.69 |                0.89 |  0.03 |                    7.5  |  2.1  |                 1.81 |  0.19 |         0.03 |  0    |    0.51 |  0.04 |
| 10 | LG             | Gastrocnemius LH          |     1031.17 |  30.53 |                1.56 |  0.14 |                   14.2  |  3.6  |                 3.42 |  0.08 |         0.62 |  0.05 |    0.46 |  0.05 |
| 11 | MG             | Gastrocnemius MH          |      849.17 |  24.18 |                1.61 |  0.13 |                   14    |  3.9  |                 3.61 |  0.06 |         0.49 |  0.04 |    0.45 |  0.04 |
| 12 | GMe            | Gluteus medius            |     1992.33 |  47.38 |                2.43 |  0.11 |                    0.6  |  0.8  |                 3.64 |  0.16 |         0.79 |  0.05 |    0.67 |  0.05 |
| 13 | CF             | Gluteus superficialis     |      521    |  31.66 |                1.92 |  0.11 |                   -1.1  |  0.7  |                 2.55 |  0.17 |         0.26 |  0.01 |    0.56 |  0.03 |
| 14 | GA             | Gracilis caudal head      |      276    |  38.68 |                2.64 |  0.08 |                    0    |  0    |                 3.47 |  0.14 |         0.1  |  0.02 |    0.76 |  0.02 |
| 15 | GP             | Gracilis cranial head     |      173    |  27.68 |                2.57 |  0.13 |                    0    |  0    |                 3.04 |  0.15 |         0.06 |  0.01 |    0.84 |  0.01 |
| 16 | nan            | Hip abductors             |     1256.67 | 735.67 |                2.18 |  0.25 |                   -0.28 |  0.83 |                 3.09 |  0.54 |         0.52 |  0.26 |    0.72 |  0.04 |
| 17 | nan            | Hip adductors             |      402.3  | 118.84 |                1.28 |  0.39 |                    0.94 |  0.81 |                 2.42 |  0.37 |         0.2  |  0.05 |    0.75 |  0.03 |
| 18 | nan            | Hip extensors             |     1300.64 | 369.73 |                3.19 |  0.4  |                    1.02 |  0.67 |                 4.3  |  0.48 |         0.41 |  0.1  |    0.74 |  0.03 |
| 19 | nan            | Hip flexors               |     1365.08 | 419.75 |                1.76 |  0.6  |                   21.25 |  4.17 |                 4.38 |  1.17 |         0.69 |  0.02 |    0.39 |  0.03 |
| 20 | nan            | Knee extensors            |      711.83 | 250.48 |                1.49 |  0.2  |                   16.77 |  4.83 |                 3.01 |  0.18 |         0.42 |  0.14 |    0.5  |  0.06 |
| 21 | nan            | Knee flexors              |      890.81 | 261.32 |                2.72 |  0.37 |                    5.69 |  2.33 |                 4.14 |  0.31 |         0.33 |  0.08 |    0.65 |  0.06 |
| 22 | MX             | Muscle X                  |      377    |  17.21 |                3.5  |  0.08 |                    1    |  1    |                 4.31 |  0.12 |         0.1  |  0.01 |    0.81 |  0.01 |
| 23 | PerB           | Peroneus brevis           |      124.67 |   4.45 |                0.82 |  0.02 |                   12.6  |  2.8  |                 2.92 |  0.11 |         0.14 |  0.01 |    0.28 |  0.01 |
| 24 | PerL           | Peroneus longus           |      177.5  |   7.01 |                0.89 |  0.08 |                   20    |  3.1  |                 2.76 |  0.04 |         0.19 |  0.02 |    0.32 |  0.03 |
| 25 | nan            | Plantarflexors†           |      371.56 | 117.83 |                1.19 |  0.15 |                   14.88 |  2.21 |                 3.04 |  0.22 |         0.27 |  0.07 |    0.4  |  0.05 |
| 26 | Pla            | Plantaris*,†              |      397.5  |  16.87 |                1.37 |  0.07 |                   16.4  |  3.2  |                 4.08 |  0.07 |         0.26 |  0.01 |    0.34 |  0.02 |
| 27 | IP             | Psoas                     |     1784.83 | 296.1  |                2.36 |  0.23 |                   17.1  |  0.8  |                 5.55 |  0.26 |         0.67 |  0.07 |    0.42 |  0.02 |
| 28 | RF             | Rectus femoris            |      945.33 |  33.23 |                1.16 |  0.1  |                   25.4  |  4    |                 3.21 |  0.11 |         0.71 |  0.05 |    0.36 |  0.04 |
| 29 | SM             | Semimembranosus           |     1452.83 |  39.84 |                3.09 |  0.13 |                    2.1  |  2    |                 4.27 |  0.12 |         0.45 |  0.02 |    0.72 |  0.02 |
| 30 | ST             | Semitendinosus            |      789.83 |  33.17 |                4.77 |  0.05 |                    0    |  0    |                 5.95 |  0.09 |         0.16 |  0.02 |    0.8  |  0.01 |
| 31 | Sol            | Soleus*,†                 |      134.67 |   4.57 |                1.97 |  0.19 |                    3.9  |  2.4  |                 2.83 |  0.19 |         0.07 |  0.01 |    0.69 |  0.04 |
| 32 | TA             | Tibialis anterior         |      662.17 |  17.7  |                1.64 |  0.07 |                   12.8  |  1.2  |                 2.91 |  0.04 |         0.38 |  0.02 |    0.57 |  0.03 |
| 33 | TP             | Tibialis posterior        |      135.83 |   9.71 |                0.6  |  0.04 |                   26    |  7.3  |                 2.67 |  0.01 |         0.19 |  0.03 |    0.23 |  0.01 |
| 34 | VI             | Vastus intermedius        |      170.67 |  15.53 |                1.14 |  0.08 |                    6.9  |  1.2  |                 2.64 |  0.16 |         0.14 |  0.01 |    0.44 |  0.04 |
| 35 | VL             | Vastus lateralis          |     1288.83 |  41.22 |                1.96 |  0.12 |                   10    |  3.3  |                 3.42 |  0.08 |         0.62 |  0.05 |    0.57 |  0.04 |
| 36 | VM             | Vastus medalis            |      442.5  |  44.81 |                1.7  |  0.04 |                   24.7  |  3.4  |                 2.79 |  0.09 |         0.22 |  0.02 |    0.61 |  0.02 |

In [15]:
model = osim.Model(out.as_posix())
muscles : osim.SetMuscles = model.getMuscles()
for i in range(muscles.getSize()):
    muscle : osim.Muscle = muscles.get(i)
    muscle_name = muscle.getName().replace('R_', '')
    params = johnson_params.row(by_predicate=pl.col('Abbreviation') == muscle_name, named=True)
    if params is None:
        print(f"Muscle {muscle_name} not found in parameters file.")
        continue
    f0 = params['Fo (N)']
    l0 = params['l0 (mm)'] / 1000  # Convert mm to m
    lts = params['lts (mm)'] / 1000  # Convert mm to m
    alpha = params['θ0 (deg)'] * np.pi / 180  # Convert deg to rad
    muscle.setMaxIsometricForce(f0)
    muscle.setOptimalFiberLength(l0)
    muscle.setTendonSlackLength(lts)
    muscle.setPennationAngleAtOptimalFiberLength(alpha)
    print(f"Updated {muscle_name}: F0={f0}, l0={l0}, lts={lts}, alpha={alpha}")

model.finalizeFromProperties()
model.finalizeConnections()
out = model_dir / 'rat_hindlimb_millard_y2j_knee_fixed_markers_params.osim'
model.printToXML(out.as_posix())

Updated BFa: F0=3.39, l0=0.039700000000000006, lts=0.0, alpha=0.0
Updated BFp: F0=12.49, l0=0.0376, lts=0.0, alpha=0.12217304763960307
Updated CF: F0=2.06, l0=0.037899999999999996, lts=0.0, alpha=0.0
Updated STp: F0=4.54, l0=0.0436, lts=0.0, alpha=0.0
Updated STa: F0=1.39, l0=0.0341, lts=0.0, alpha=0.0
Updated SM: F0=10.22, l0=0.0357, lts=0.0, alpha=0.0
Updated QF: F0=3.0, l0=0.011, lts=0.0, alpha=0.05235987755982988
Updated TFL: F0=3.89, l0=0.0206, lts=0.0, alpha=0.19198621771937624
Updated GMa: F0=9.23, l0=0.019600000000000003, lts=0.0048, alpha=0.20943951023931953
Updated GMe: F0=18.74, l0=0.019899999999999998, lts=0.0, alpha=0.20943951023931953
Updated GMi: F0=7.39, l0=0.011800000000000001, lts=0.0, alpha=0.2792526803190927
Updated Pir: F0=11.14, l0=0.0108, lts=0.0, alpha=0.2792526803190927
Updated GP: F0=2.35, l0=0.0293, lts=0.0009, alpha=0.05235987755982988
Updated GA: F0=1.55, l0=0.0303, lts=0.0, alpha=0.0
Updated AL: F0=1.039, l0=0.0166, lts=0.0045, alpha=0.15707963267948966
Up

True

In [16]:
from osimpy import OsimGraph
from tsl_optimization import optimize_fiber_length, calc_tsl

model = osim.Model(out.as_posix())
graph = OsimGraph(osim_model=model)
lm_norm_range = (0.5, 1.5) # Normalized fiber length range for optimization

results_full = graph.get_all_muscle_lengths_rom(min_points=100)
for muscle_name, lmt in results_full.items():
  muscle: osim.Muscle = graph.get_muscle(muscle_name) 
  lm_opt = float(muscle.get_optimal_fiber_length())
  lm_range = lm_opt * np.asarray(lm_norm_range)
  alpha_opt = float(muscle.get_pennation_angle_at_optimal())
  lmt = np.clip(np.sort(np.unique(lmt.select(pl.col(muscle_name)))), 1.0e-6, None) # Ensure lmt is sorted, unique, and non-zero
  
  millard: osim.Millard2012EquilibriumMuscle = osim.Millard2012EquilibriumMuscle.safeDownCast(muscle)
  afl = millard.getActiveForceLengthCurve()
  pfl = millard.getFiberForceLengthCurve()
  tfl = millard.getTendonForceLengthCurve()
  lm = optimize_fiber_length(lmt, lm_opt, alpha_opt, afl, pfl, tfl, lm_norm_range)
  tsl = calc_tsl(lmt, lm, lm_opt, alpha_opt, afl, pfl, tfl)

In [17]:
#| label: tbl-tsl-comparison

# Load Johnson params

# Compare original and optimized tendon slack lengths

In [18]:
model = osim.Model(out.as_posix())
# TODO: Turn this into functions
body_set: osim.BodySet = model.getBodySet()
cloned_bodies = {}
n_bodies = body_set.getSize()
for i in range(n_bodies):
    body: osim.Body = model.getBodySet().get(i)
    body_name: str = body.getName()
    new_body_name = body_name
    if body_name != 'spine': # Spine does not get mirrored
        new_body: osim.Body = body.clone()
        new_body_name = body_name.replace('_r', '_l')
        new_body.setName(new_body_name)
        geom: osim.Geometry = new_body.get_attached_geometry(0)
        geom.setName(geom.getName().replace('_r', '_l'))
        geom.set_scale_factors(osim.Vec3(1, 1, -1))
        com: osim.Vec3 = new_body.getMassCenter()
        new_body.setMassCenter(osim.Vec3([com.get(0), com.get(1), -com.get(2)]))
        moi: osim.Inertia = new_body.getInertia()
        moments: osim.Vec3 = moi.getMoments()
        products: osim.Vec3 = moi.getProducts()
        new_body.setInertia(osim.Inertia(
                            moments.get(0), moments.get(1), moments.get(2),
                            products.get(0), -products.get(1), -products.get(2)))
        model.addBody(new_body)
    cloned_bodies[body_name] = new_body_name

joint_set: osim.JointSet = model.getJointSet()
left_joint_set: osim.JointSet = joint_set.clone()
# Remove ground_spine
left_joint_set.remove(left_joint_set.get('ground_spine'))

left_joint_set_file = './models/osim/left_joint_set.xml'
left_joint_set.printToXML(left_joint_set_file)
with open(left_joint_set_file, 'r') as file:
    joint_set_content = file.read()

for old_body_name, new_body_name in cloned_bodies.items():
    joint_set_content = joint_set_content.replace(old_body_name, new_body_name)

new_joint_names = []
for i in range(left_joint_set.getSize()):
    joint: osim.Joint = left_joint_set.get(i)
    joint_name: str = joint.getName()
    new_joint_name: str = joint_name.replace('_r', '_l')
    joint_set_content = joint_set_content.replace(joint_name, new_joint_name)
    new_joint_names.append(new_joint_name)
    
with open(left_joint_set_file, 'w') as file:
    file.write(joint_set_content)
    
left_joint_set = osim.JointSet(left_joint_set_file)
# Remove the left_joint_set file
import os
os.remove(left_joint_set_file)
    
js: osim.JointSet = model.getJointSet()
for i in range(left_joint_set.getSize()):
    js.cloneAndAppend(left_joint_set.get(i))

model.initSystem()
for joint_name in new_joint_names:
    joint: osim.CustomJoint = osim.CustomJoint.safeDownCast(model.getJointSet().get(joint_name))
    if joint is None:
        print(f"Joint {joint_name} not found in the model.")
        continue
    # Mirror the parent and child frames
    parent_offset: osim.PhysicalOffsetFrame = osim.PhysicalOffsetFrame.safeDownCast(joint.getParentFrame())
    parent_pos = parent_offset.get_translation()
    parent_rot = parent_offset.get_orientation()
    child_offset: osim.PhysicalOffsetFrame = osim.PhysicalOffsetFrame.safeDownCast(joint.getChildFrame())
    child_pos = child_offset.get_translation()
    child_rot = child_offset.get_orientation()
    parent_offset.set_translation(osim.Vec3(parent_pos.get(0), parent_pos.get(1), -parent_pos.get(2)))
    child_offset.set_translation(osim.Vec3(child_pos.get(0), child_pos.get(1), -child_pos.get(2)))
    
    # Mirror the spatial transform
    spatial_transform: osim.SpatialTransform = joint.get_SpatialTransform()
    transform_axes: list[osim.Vec3] = spatial_transform.getAxes()
    for i, vec in enumerate(transform_axes):
        # For rotations, negate x and y components
        # For translations, negate z component
        if i <= 2 and vec[2]:  # Skip the first three axes (rotation1-3)
            continue
        elif i > 2 and not vec[2]:  # Skip the first three axes (translation1-3)
            continue
            
        # Assume that order is always rotation1-3, translation1-3
        get_transform_function = ('get_rotation' + str(i+1)) if i < 3 else ('get_translation' + str(i-2))
        transform_axis: osim.TransformAxis = getattr(spatial_transform, get_transform_function)()
        axis_function: osim.Function = transform_axis.getFunction()
        concreteClass = axis_function.getConcreteClassName()
        match concreteClass:
            case 'SimmSpline':
                # Mirror the function
                simm_spline: osim.SimmSpline = osim.SimmSpline.safeDownCast(axis_function)
                if simm_spline is None:
                    raise ValueError("Function is not a SimmSpline.")
                for k in range(simm_spline.getSize()):
                    y: float = simm_spline.getY(k)
                    simm_spline.setY(k, -y)
            case 'LinearFunction':
                linear_func: osim.LinearFunction = osim.LinearFunction.safeDownCast(axis_function)
                if linear_func is None:
                    raise ValueError("Function is not a LinearFunction.")
                linear_func.setSlope(-linear_func.getSlope())
                linear_func.setIntercept(-linear_func.getIntercept())
            case 'Constant':
                constant_func: osim.Constant = osim.Constant.safeDownCast(axis_function)
                if constant_func is None:
                    raise ValueError("Function is not a Constant.")
                constant_func.setValue(-constant_func.getValue())
            case 'MultiplierFunction':
                mult_func: osim.MultiplierFunction = osim.MultiplierFunction.safeDownCast(axis_function)
                if mult_func is None:
                    raise ValueError("Function is not a MultiplierFunction.")
                # For MultiplierFunction, negate the scale
                mult_func.setScale(-mult_func.getScale())
            case _:
                print(f"Unsupported function type: {concreteClass}, not mirroring")

# Update muscles 
muscle_set: osim.ForceSet = model.getForceSet()
left_muscle_set_file = 'models/osim/left_muscle_set.xml'
muscle_set.printToXML(left_muscle_set_file)
with open(left_muscle_set_file, 'r') as file:
    muscle_set_content = file.read()

for old_body_name, new_body_name in cloned_bodies.items():
    muscle_set_content = muscle_set_content.replace(old_body_name, new_body_name)
    
new_muscle_names = []
for i in range(muscle_set.getSize()):
    muscle: osim.Muscle = muscle_set.get(i)
    muscle_name: str = muscle.getName()
    new_muscle_name: str = muscle_name.replace('R_', 'L_')
    muscle_set_content = muscle_set_content.replace(muscle_name, new_muscle_name)
    new_muscle_names.append(new_muscle_name)

for old_body_name, new_body_name in cloned_bodies.items():
    muscle_set_content = muscle_set_content.replace(old_body_name, new_body_name)

with open(left_muscle_set_file, 'w') as file:
    file.write(muscle_set_content)
    
left_muscle_set = osim.ForceSet(left_muscle_set_file)
os.remove(left_muscle_set_file)

for muscle_name in new_muscle_names:
    muscle: osim.Muscle = osim.Muscle.safeDownCast(left_muscle_set.get(muscle_name))
    path_points: osim.PathPointSet = muscle.getGeometryPath().getPathPointSet()
    for i in range(path_points.getSize()):
        path_point: osim.PathPoint = osim.PathPoint.safeDownCast(path_points.get(i))
        loc: osim.Vec3 = path_point.get_location()
        path_point.setLocation(osim.Vec3(loc.get(0), loc.get(1), -loc.get(2)))
    muscle_set.append(muscle)

# Make sure to update the model before printing
model.finalizeFromProperties()
model.finalizeConnections()

# For some reason this has to happen after finalizing connections
marker_set = osim.MarkerSet('models/osim/rat_hindlimb_bilateral_markerset.xml')
model.getMarkerSet().clearAndDestroy()
model.updateMarkerSet(marker_set)

model.printToXML('models/osim/rat_hindlimb_bilateral.osim')

True